In [1]:
import argparse

import cv2
import numpy as np
import torch
from torch import nn
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam import GradCAM, \
    ScoreCAM, \
    GradCAMPlusPlus, \
    AblationCAM, \
    XGradCAM, \
    EigenCAM, \
    EigenGradCAM, \
    LayerCAM, \
    FullGrad

from pytorch_grad_cam.utils.image import show_cam_on_image, \
    preprocess_image
from pytorch_grad_cam.ablation_layer import AblationLayerVit
import clip

In [2]:
def reshape_transform(tensor, height=16, width=16):
    result = tensor.transpose(0, 1)[:, 1:, :].reshape(tensor.transpose(0, 1).size(0),
                                      height, width, tensor.size(2))
    # Bring the channels to the first dimension,
    # like in CNNs.
    result = result.transpose(2, 3).transpose(1, 2)
    return result

In [3]:
class CLIPImageClassifier(nn.Module):
    def __init__(self, labels):
        super(CLIPImageClassifier, self).__init__()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model, _ = clip.load("ViT-L/14", device=self.device)
        self.labels = labels

    def forward(self, image):
        text_list_prompt = [f"This is a photo of {item}." for item in self.labels]
        text = clip.tokenize(text_list_prompt).to(self.device)
        text_features = self.clip_model.encode_text(text)
        image_features = self.clip_model.encode_image(image)

        # normalized features
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)
        logits_per_image = 100. * image_features @ text_features.t()
        
        probs = logits_per_image.softmax(dim=1)

        for label, prob in zip(self.labels, probs[0]):
            print(f"{label}: {prob:.4f}")
        return probs

In [40]:
labels = ['zoo', 'person']

model = CLIPImageClassifier(labels).to(torch.device("cuda" if torch.cuda.is_available() else "cpu")).eval()
print(model)

target_layers = [model.clip_model.visual.transformer.resblocks[-1].ln_1]

cam = GradCAM(model=model,
            target_layers=target_layers,
            reshape_transform=reshape_transform)

CLIPImageClassifier(
  (clip_model): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
      (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): Sequential(
          (0): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
            )
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
              (gelu): QuickGELU()
              (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
            )
            (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          )
          (1): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj): Non

In [41]:
# 读取输入图像
image_path = "/Users/yifanlu/Desktop/zoo2.jpg"

rgb_img = cv2.imread(image_path, 1)[:, :, ::-1]
rgb_img = cv2.resize(rgb_img, (224, 224))

# 预处理图像
rgb_img = np.float32(rgb_img) / 255
input_tensor = preprocess_image(rgb_img, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

# 计算 grad-cam
targets = [ClassifierOutputTarget(labels.index('zoo'))] # 可以指定一个类别，或者使用 None 表示最高概率的类别
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
grayscale_cam = grayscale_cam[0, :]

# 将 grad-cam 的输出叠加到原始图像上
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# 保存可视化结果
cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR, visualization)
cv2.imwrite('cam_try.jpg', visualization)

zoo: 0.9998
person: 0.0002


True

----------

In [42]:
class DualCLIPImageClassifier(nn.Module):
    def __init__(self, labels):
        super(DualCLIPImageClassifier, self).__init__()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model, _ = clip.load('ViT-L/14', device=self.device)
        self.labels = labels

    def forward(self, image):
        text_list_prompt = [f"This is a photo {item}." for item in self.labels]
        text = clip.tokenize(text_list_prompt).to(self.device)
        text_features = self.clip_model.encode_text(text)
        image_features = self.clip_model.encode_image(image)
        # num_attributs = len(self.labels) / 2
        
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)
        cs = image_features @ text_features.T
        cs_softmax = torch.softmax(cs.reshape([-1, 2, 2]), dim=2)
        logits_per_image = cs_softmax.reshape([-1, 2*2])

        for label, logit in zip(self.labels, logits_per_image[0]):
            print(f"{label}: {logit:.4f}")
        return logits_per_image

In [43]:
labels = ['of zoo', 'without zoo', 'of person', 'without person']
model = DualCLIPImageClassifier(labels).to(torch.device("cuda" if torch.cuda.is_available() else "cpu")).eval()
print(model)

target_layers = [model.clip_model.visual.transformer.resblocks[-1].ln_1]

cam = GradCAM(model=model,
            target_layers=target_layers,
            reshape_transform=reshape_transform)

DualCLIPImageClassifier(
  (clip_model): CLIP(
    (visual): VisionTransformer(
      (conv1): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
      (ln_pre): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (transformer): Transformer(
        (resblocks): Sequential(
          (0): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=1024, out_features=1024, bias=True)
            )
            (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): Sequential(
              (c_fc): Linear(in_features=1024, out_features=4096, bias=True)
              (gelu): QuickGELU()
              (c_proj): Linear(in_features=4096, out_features=1024, bias=True)
            )
            (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          )
          (1): ResidualAttentionBlock(
            (attn): MultiheadAttention(
              (out_proj):

In [44]:
image_path = "/Users/yifanlu/Desktop/zoo2.jpg"

rgb_img = cv2.imread(image_path, 1)[:, :, ::-1]
rgb_img = cv2.resize(rgb_img, (224, 224))

# 预处理图像
rgb_img = np.float32(rgb_img) / 255
input_tensor = preprocess_image(rgb_img, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

# 计算 grad-cam
target_category = [ClassifierOutputTarget(labels.index('of zoo'))] # 可以指定一个类别，或者使用 None 表示最高概率的类别
grayscale_cam = cam(input_tensor=input_tensor, targets=target_category)
grayscale_cam = grayscale_cam[0, :]

# 将 grad-cam 的输出叠加到原始图像上
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# 保存可视化结果
cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR, visualization)
cv2.imwrite('cam_dual.jpg', visualization)

of zoo: 0.5149
without zoo: 0.4851
of person: 0.4975
without person: 0.5025


True